# VZP: real invitations to mammography screening

One row is one invitation, not an independent woman. The extract is limited to women continuously
insured with VZP in 2015–2025; transferability to other groups is untested. This notebook explores
associations and plans the experiment; it does not train the model.

Run: `pip install -e ".[dev]"`, put the CSV at the path from the config and run all cells. In the IDE,
select the Python kernel from the project `.venv`. The first code cell finds the repository root and
adds the local `src` to the import path, so the notebook also runs from the `notebooks` folder without
an editable install. Set `SCREENING_EDA_DATA` to use another local file.
All tables and PNGs are saved to the ignored `reports/` folder; the notebook is committed without outputs.
Never save or commit rows with identifiers.

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/screening/__init__.py').is_file()
             and (p / 'configs/config.yaml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or its notebooks directory.')
sys.path.insert(0, str(ROOT / 'src'))
print(f'Python kernel: {sys.executable}')
print(f'Project root: {ROOT}')


In [ ]:
import os

import pandas as pd
from IPython.display import Image, display

from screening.config import load_config
from screening.evaluation.real_data import (
    build_tables,
    make_figures,
    read_delivery,
    save_tables,
)

cfg = load_config(ROOT / 'configs/config.yaml')
source = Path(os.environ.get('SCREENING_EDA_DATA', cfg['data']['path']))
if not source.is_absolute():
    source = ROOT / source
if not source.is_file():
    raise FileNotFoundError('Provide the VZP CSV locally; raw data must never be committed.')
report_dir = ROOT / 'reports/eda'
figure_dir = ROOT / 'reports/figures'


## Checks and unit of observation

`SCREENING_3M/6M/12M_FLAG` are outcomes after the invitation, never model inputs.
EDA uses attendance (`A = 1`); the model predicts the opposite outcome, non-attendance.
Dates are parsed strictly as `%d.%m.%Y`. We check identifiers, one row per woman per year
and nesting of the cumulative outcomes: 3M ≤ 6M ≤ 12M.

Closure of the 365-day window must be confirmed by complete claims reporting;
similar attendance across years alone does not prove complete follow-up.

In [ ]:
df = read_delivery(source)
display(pd.DataFrame([{
    'invitations': len(df), 'clients': df.PERSON_ID.nunique(),
    'date_min': df.date.min(), 'date_max': df.date.max(),
    'attendance_365d': df.attended_12m.mean(),
    'first_flag_counter_disagreements': int((df.FIRST_INVITATION.eq('A') != df.PRIOR_INVITATION_COUNT.eq(1)).sum()),
}]))
display(df.isna().sum().rename('missing').to_frame())


## Shared reproducible tables

Rates over all years are weighted by invitations; 95% intervals account for repeated rows of the
same woman via cluster-robust variance. Single-year groups with one row per woman use Wilson
intervals. Intervals are pointwise, not simultaneous.

The territory list for the regional hypothesis is fixed on 2020–2022 (`eda` in the config) and then
applied to 2024. The 2024 ranking is not used to choose these territories.

In [ ]:
tables = build_tables(df, cfg)
save_tables(tables, report_dir, source, cfg)
figures = make_figures(tables, figure_dir, cfg['split']['test_year'])
display(tables['by_year'])
print(f'Saved {len(tables)} aggregate tables and {len(figures)} PNG figures.')


## Invitations, age and contact with healthcare

The invitation counter is shown as delivered, including the current invitation.
First invitations are counted separately from the delivered `FIRST_INVITATION` flag.
Procedure counts are not divided by two: the mapping of codes to visits is not confirmed.
Age, invitation history and prevention are correlated; these are not causal effects.

In [ ]:
display(tables['first_invitation'])
for name in ['invitations', 'age', 'contacts', 'gyn_prevention', 'other_screening']:
    display(Image(filename=str(figure_dir / f'{name}.png'), width=900))


## 75 VZP territorial offices

A VZP territorial office is not the exact place of residence nor a measured accessibility.
The ranking shows crude attendance without adjustment for age or population mix.
Overlapping intervals and 75 comparisons do not allow declaring individual places a causal
"geographic barrier". The full table including `n` is stored as CSV.

In [ ]:
display(tables['regions_reporting_year'])
display(tables['regional_reference'].loc[lambda x: x.selected])
display(Image(filename=str(figure_dir / 'regions_ranking.png'), width=950))


## Response speed: 90 / 180 / 365 days

Flags are cumulative. The 6M−3M difference is the share of invitations whose first recorded screening
falls between these checkpoints; 12M−6M covers day 180 to day 365.
There is no exact examination date or optimal reminder day here.
A reminder, e.g. after 90 days, remains a hypothesis for a randomised test.

In [ ]:
timing = tables['response_timing'].copy()
timing['additional_attendance_since_previous_horizon'] = timing.attendance.diff().fillna(timing.attendance)
display(timing)
display(Image(filename=str(figure_dir / 'response_timing.png'), width=900))


## Invitations shortly after a recorded screening

Windows 0–1 and 0–6 months are counted separately. They are nested and must not be added up.
These are candidates for reviewing the mailing process, not proven unnecessary invitations.
Months may be rounded; zero alone does not prove a date error.

`eda.letter_cost_czk` is an explicitly assumed cost per letter. Letters × cost is a gross scenario
cost, not confirmed savings. This analysis does not justify cancelling invitations; record checks and
approval by the programme owner are required.

In [ ]:
display(tables['recent_screening_cost'])
display(Image(filename=str(figure_dir / 'recent_screening_cost.png'), width=1000))


## Segment hypotheses and experiment design

Hypothesis tables overlap and do not add up. For A/B planning we use separate, non-overlapping 2024
groups with the priority order of the production code. The planning group `regional` is formed only
from the remaining `other`; the production rules are not changed by this notebook.
Recently screened women are not included in any plan to intensify invitations.

Group sizes come from `experiment.power.n_per_group`: independent women, two arms 1:1, two-sided
alpha=0.05, power=0.80, absolute MDEs from the config. These are separate A/B options, not a ready
multi-arm test with a shared control. The 2024 size is an upper bound for a yearly wave, not a promise
of the recruitment available now.

In [ ]:
display(tables['hypotheses_all_years'])
display(tables['hypotheses_reporting_year'])
display(tables['segments_exclusive'])
display(tables['experiment_sizes'])
assert tables['segments_exclusive'].n.sum() == df.year.eq(cfg['split']['test_year']).sum()
assert tables['segments_exclusive'].n.eq(tables['segments_exclusive'].clients).all()


## Still needed before an A/B test

- Fix the intervention, eligibility criteria and one record per woman per experiment.
- Use real predictions of the frozen model for risk quintiles; this EDA does not invent them.
- Randomise 1:1 within segment × quintile; store assignments and the seed.
- Primary outcome: attendance within 365 days; 90 days is a secondary early signal, not a replacement.
- ITT analysis: do not drop unreachable women or undelivered messages. An unknown outcome is not zero.
- One final analysis; early stops only for safety/data quality, no daily search for p<0.05.
- Recalculate sizes for multiplicity before several confirmatory comparisons.

Details and tables: `docs/segments-real-data.md`, `docs/experiment-design.md`.